# Paper3 — complete revised replication

This notebook rebuilds survey linkage, historical OSM extraction, all statistical tables, four figures, the 50-reference manuscript, and the LaTeX/PDF package. Run cells in order.

**Requirements:** Python 3.12 and `revision_20260920/requirements.txt`; Pandoc and XeLaTeX on PATH. Full replication requires authorized access to the original and cleaned survey files and both PBFs. The public repository contains aggregates, not the restricted inputs. `FULL_REBUILD=False` rebuilds the manuscript from supplied aggregate outputs and figures only.

The map uses 500 m cells with n >= 5; no exact respondent points are displayed. The routing illustration uses a synthetic origin selected only from public street geometry. Neither overlap of distributions nor non-significance proves equivalence.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib
candidates=[Path.cwd(),Path.cwd()/'ttll-p3',Path.cwd()/'Paper3'/'ttll-p3']
REPO=next((x.resolve() for x in candidates if (x/'revision_20260920').is_dir()),None)
if REPO is None: raise RuntimeError('Open this notebook from the ttll-p3 repository root.')
REV=REPO/'revision_20260920'
FULL_REBUILD=True
BUILD_PDF=True
# Override environment variables below or edit these paths for another machine.
WORKSPACE=Path(os.environ.get('TTLL_WORKSPACE',str(REPO.parents[1])))
CLEAN=Path(os.environ.get('TTLL_CLEAN_SURVEY',str(WORKSPACE/'Data'/'QData_Citizens.xlsx')))
raw_candidates=list((WORKSPACE/'Data').glob('*người dân*'))
RAW=Path(os.environ.get('TTLL_RAW_SURVEY',str(raw_candidates[0] if raw_candidates else WORKSPACE/'Data'/'original_resident_responses.xlsx')))
PBF=Path(os.environ.get('TTLL_PBF_DIR',str(WORKSPACE/'Paper3'/'DataPaper3'/'dataset')))
CACHE=Path(os.environ.get('TTLL_CACHE_DIR',str(REV/'restricted_cache')))
ANALYSIS_PYTHON=os.environ.get('TTLL_ANALYSIS_PYTHON',sys.executable)
BUILD_PYTHON=os.environ.get('TTLL_BUILD_PYTHON',sys.executable)
LATEX_OUT=Path(os.environ.get('TTLL_LATEX_OUT',str(REPO/'paper'/'revised')))
def run(args):
    result=subprocess.run([str(x) for x in args],cwd=REPO,capture_output=True,text=True,encoding='utf-8',errors='replace')
    if result.returncode: raise RuntimeError(result.stdout[-4000:]+'\n'+result.stderr[-4000:])
    print(result.stdout[-600:])
print('Configured full rebuild:',FULL_REBUILD)


## 1. Source checks and full analysis
Hashes of restricted source files and PBFs are recorded in `analysis/results.json`. Cached OSM extraction is reused only when its PBF hash matches. The 2026 snapshot is a mapping sensitivity, not a metro impact estimate. First extraction can take several minutes.

In [ ]:
if FULL_REBUILD:
    required=[CLEAN,RAW,PBF/'vietnam-230101.osm.pbf',PBF/'vietnam-260101.osm.pbf']
    missing=[str(x) for x in required if not x.is_file()]
    if missing: raise FileNotFoundError('Provide authorized input paths: '+', '.join(missing))
    run([ANALYSIS_PYTHON,REV/'analyse_paper3.py','--clean-survey',CLEAN,'--raw-survey',RAW,'--pbf-dir',PBF,'--cache-dir',CACHE,'--make-maps'])
else:
    print('Aggregate-only mode: no fresh survey or spatial analysis was performed.')


## 2. Inspect aggregate findings and map disclosure checks

In [ ]:
import pandas as pd
from IPython.display import display, Image
for name in ['reasons','distances_by_reason','walking_contrasts','adjusted_associations','cohort_sensitivity','omnibus_tests']:
    print(name)
    display(pd.read_csv(REV/'analysis'/f'{name}.csv'))
audit=json.loads((REV/'analysis/map_audit.json').read_text(encoding='utf-8'))
assert not audit['exact_respondent_points_published']
assert audit['illustration_origin_connector_m']<=100
assert audit['illustration_stop_connector_m']<=100
print(audit)


In [ ]:
for name in ['Figure1_study_area.png','Figure2_network_method.png','Figure2_accessibility.png','Figure1_reasons.png']:
    display(Image(filename=str(REV/'analysis'/name),width=900))


## 3. Build Word, Markdown, references and LaTeX/PDF
The PDF below is compiled from LaTeX. Word is also rebuilt, but its page layout is not guaranteed identical to LaTeX. Review the PDF visually after changes.

In [ ]:
run([BUILD_PYTHON,REV/'build_manuscript.py'])
args=[BUILD_PYTHON,REV/'build_latex.py','--output-dir',LATEX_OUT]
if not BUILD_PDF: args.append('--no-compile')
run(args)
run([BUILD_PYTHON,REV/'verify_package.py'])


## 4. Audit output provenance
This manifest records public output hashes only. Do not upload survey workbooks, precise coordinates, local caches or notebook outputs containing private records. This notebook is stored with outputs cleared in Git.

In [ ]:
files=list((REV/'analysis').glob('*.csv'))+list((REV/'analysis').glob('*.png'))+[LATEX_OUT/'Paper3_Revised.tex']
if BUILD_PDF: files.append(LATEX_OUT/'Paper3_Revised.pdf')
manifest={x.name:hashlib.sha256(x.read_bytes()).hexdigest() for x in files}
(REV/'reproduction_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('Completed:',len(manifest),'output hashes recorded.')
